In [5]:
# Call all functions used in starter file
# NOTE: Change to local path when using this script
%run "C:\Users\gabri\Documents\Chicago\Palm_watch\PalmWatch\deforestation_data\ImportFilesAndFunctions.ipynb"

Seed set to 42


All imports successful!


In [6]:
# load the dataset
palmoil_df = csv_old

# check for any null values
palmoil_df[palmoil_df.isnull().any(axis=1)]

# fills all empty/na/nan values (usually provinces with no neighbors) with 0
palmoil_df.fillna(0, inplace=True)

In [7]:
# drop years that we need to predict
palmoil_wo_target = palmoil_df[palmoil_df['year'] <= 2018]

In [8]:
# list with features and targets
feature_names = ['defor_frac', 'degr_frac', 'remaining_frac_baseline', 'remaining_frac_hex', 'remaining_forest_ha', 'nbr_r1_defor_frac', 'nbr_r2_defor_frac' , 'nbr_r1_remaining_frac_baseline', 'target_total_5yr' , 'target_palm_5yr']

# colloquial names for every feature
common_names = ['Fraction of Deforestation', 'Fraction of Degradation', 'Cumulative Remaining Forest', 'Current Forest Density', 'Remaining Forest (hectacres)', 'Neighboring 6 Deforestation Fraction', 'Neighboring 12 Deforestation Fraction', 'Average Neighboring Remaining Forest' , 'Predicted 5 Year Deforestation', 'Predicted 5 Year Deforestation for Palm Oil']

# list of all provinces
provinces = ['Aceh', 'SumateraUtara', 'Riau', 'SumateraBarat', 'Lampung', 'SumateraSelatan', 'Jambi', 'Bengkulu']

# year range of our dataset
YEARS = range(2000, 2019)

In [9]:
# function to create a dataframe of a specific province and year
def sortByProvince(province, df, year):
  """Create a new dataframe and return all rows of a specific province and specific year"""
  new_df = df[(df['province'] == province) & (df['year'] == year)]

  return new_df

In [10]:
# function to loop through all features and find the median of a group of 16 hex id's
def create_subsetted_data(province, df, year, feature_names, group_size=16):
  """Creates a new dataframe that finds the median of a group of 16 hex_id"""
  province_df = sortByProvince(province, df, year)

  subsetted_data = {
      "Grouped ID": [],
      "Year": []
  }

  for feature in feature_names:
      subsetted_data[f"{feature}_median"] = []

  for i in range(0, len(province_df), group_size):
      subsetted_data["Grouped ID"].append(i // group_size)
      subsetted_data["Year"].append(year)

      for feature in feature_names:
          subsetted_data[f"{feature}_median"].append(
              province_df[feature].iloc[i:i+group_size].median()
          )

  return pd.DataFrame(subsetted_data)

In [11]:
# creates a dataframe with all years of a specific province
def create_all_years_data(province, df, years, feature_names, group_size=16):
  """Creates a dataframe with all years of a specific province"""
  all_data = []
  df = df.sort_values(['year', 'province', 'lat', 'lon'])

  for year in years:
      yearly_df = create_subsetted_data(
          province,
          df,
          year,
          feature_names,
          group_size
      )
      all_data.append(yearly_df)

  return pd.concat(all_data, ignore_index=True)

In [12]:
# create a new dataframe province by province of a larger grouped area

# creates a dataframe with aceh
combined_aceh = create_all_years_data(
    province="Aceh",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_aceh.insert(2, 'Province', 'Aceh')

# creates a dataframe with sumatera utara
combined_sumaterautara = create_all_years_data(
    province="SumateraUtara",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_sumaterautara.insert(2, 'Province', 'SumateraUtara')

# creates a dataframe with riau
combined_riau = create_all_years_data(
    province="Riau",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_riau.insert(2, 'Province', 'Riau')

# creates a dataframe with sumatera barat
combined_sumaterabarat = create_all_years_data(
    province="SumateraBarat",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_sumaterabarat.insert(2, 'Province', 'SumateraBarat')

# creates a dataframe with lampung
combined_lampung = create_all_years_data(
    province="Lampung",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_lampung.insert(2, 'Province', 'Lampung')

# creates a dataframe with sumatera selatan
combined_sumateraselatan = create_all_years_data(
    province="SumateraSelatan",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_sumateraselatan.insert(2, 'Province', 'SumateraSelatan')

# creates a dataframe with jambi
combined_jambi = create_all_years_data(
    province="Jambi",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_jambi.insert(2, 'Province', 'Jambi')

# creates a dataframe with bengkulu
combined_bengkulu = create_all_years_data(
    province="Bengkulu",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_bengkulu.insert(2, 'Province', 'Bengkulu')

In [13]:
def create_lag_features(df, features, lags=2):
  df = df.sort_values("year").copy()

  for feature in features:
      for lag in range(1, lags+1):
          df[f"{feature}_lag{lag}"] = df[feature].shift(lag)

  return df

In [14]:
aceh = palmoil_wo_target[palmoil_wo_target['province'] == 'Aceh']
aceh_18yr = aceh.groupby('year', as_index=False)[feature_names].mean()

sumaterautara = palmoil_wo_target[palmoil_wo_target['province'] == 'SumateraUtara']
sumaterautara_18yr = sumaterautara.groupby('year', as_index=False)[feature_names].mean()

riau = palmoil_wo_target[palmoil_wo_target['province'] == 'Riau']
riau_18yr = riau.groupby('year', as_index=False)[feature_names].mean()

sumaterabarat = palmoil_wo_target[palmoil_wo_target['province'] == 'SumateraBarat']
sumaterabarat_18yr = sumaterabarat.groupby('year', as_index=False)[feature_names].mean()

lampung = palmoil_wo_target[palmoil_wo_target['province'] == 'Lampung']
lampung_18yr = lampung.groupby('year', as_index=False)[feature_names].mean()

sumateraselatan = palmoil_wo_target[palmoil_wo_target['province'] == 'SumateraSelatan']
sumateraselatan_18yr = sumateraselatan.groupby('year', as_index=False)[feature_names].mean()

jambi = palmoil_wo_target[palmoil_wo_target['province'] == 'Jambi']
jambi_18yr = jambi.groupby('year', as_index=False)[feature_names].mean()

bengkulu = palmoil_wo_target[palmoil_wo_target['province'] == 'Bengkulu']
bengkulu_18yr = bengkulu.groupby('year', as_index=False)[feature_names].mean()

In [15]:
full_provinces = [aceh, sumaterautara, riau, sumaterabarat, lampung, sumateraselatan, jambi, bengkulu]

full_provinces_18yr = [aceh_18yr, sumaterautara_18yr, riau_18yr, sumaterabarat_18yr, lampung_18yr, sumateraselatan_18yr, jambi_18yr, bengkulu_18yr]

In [25]:
rf_palm_forecasts = {}
rf_palm = RandomForestRegressor(n_estimators=195, max_depth=7, random_state=1)

for i, province in enumerate(full_provinces):

    # ================= PALM OIL RF MODEL =================

    df_palm = create_lag_features(province, feature_names, lags=2)
    df_palm.fillna(0, inplace=True)

    train_palm = df_palm[df_palm["year"] <= 2012].copy()

    lag_features = [col for col in df_palm.columns if "lag" in col]

    X_train = train_palm[lag_features]
    y_train = train_palm["target_palm_5yr"]

    rf_palm.fit(X_train, y_train)


    # recursive palm forecast
    palm_forecast = train_palm.copy()
    palm_predictions = []

    for year in range(2013, 2024):

        X_next = palm_forecast[lag_features].tail(1)

        pred = rf_palm.predict(X_next)[0]

        palm_predictions.append(pred)

        new_row = palm_forecast.tail(1).copy()

        new_row["year"] = year
        new_row["target_palm_5yr"] = pred

        for feature in feature_names:
            new_row[f"{feature}_lag2"] = palm_forecast[feature].iloc[-2]
            new_row[f"{feature}_lag1"] = palm_forecast[feature].iloc[-1]

        palm_forecast = pd.concat(
            [palm_forecast, new_row],
            ignore_index=True
        )


    # save palm dataframe
    palm_actual = full_provinces_18yr[i].sort_values("year")

    rf_palm_df = pd.DataFrame({
        "Year": range(2000, 2024),
        "Palm Oil Forecast": (
            palm_actual.loc[palm_actual["year"] <= 2012, "target_palm_5yr"].tolist()
            + palm_predictions
        )
    })

    rf_palm_forecasts[provinces[i]] = rf_palm_df

In [26]:
rf_total = RandomForestRegressor(n_estimators=195, max_depth=7, random_state=1)
rf_total_forecasts = {}
for i, province in enumerate(full_provinces):

    # ================= TOTAL DEFORESTATION RF MODEL =================
    
        df_total = create_lag_features(province, feature_names, lags=2)
        df_total.fillna(0, inplace=True)
    
        train_total = df_total[df_total["year"] <= 2018].copy()
    
        X_train = train_total[lag_features]
        y_train = train_total["target_total_5yr"]
    
        rf_total.fit(X_train, y_train)
    
    
        total_forecast = train_total.copy()
        total_predictions = []
    
    
        for year in range(2019, 2024):
    
            X_next = total_forecast[lag_features].tail(1)
    
            pred = rf_total.predict(X_next)[0]
    
            total_predictions.append(pred)
    
            new_row = total_forecast.tail(1).copy()
    
            new_row["year"] = year
            new_row["target_total_5yr"] = pred
    
            for feature in feature_names:
                new_row[f"{feature}_lag2"] = total_forecast[feature].iloc[-2]
                new_row[f"{feature}_lag1"] = total_forecast[feature].iloc[-1]
    
            total_forecast = pd.concat(
                [total_forecast, new_row],
                ignore_index=True
            )
    
    
        # save total dataframe
        total_actual = full_provinces_18yr[i].sort_values("year")
    
        rf_total_df = pd.DataFrame({
            "Year": range(2000, 2024),
            "Total Deforestation Forecast": (
                total_actual.loc[total_actual["year"] <= 2018, "target_total_5yr"].tolist()
                + total_predictions
            )
        })
    
        rf_total_forecasts[provinces[i]] = rf_total_df
    

## PALM OIL DEFORESTATION DATAFRAMES

In [ ]:
for province in provinces:
  print(province)
  display(rf_palm_forecasts[province])

## TOTAL DEFORESTATION FORECAST

In [ ]:
for province in provinces:
  print(province)
  display(rf_total_forecasts[province])